# Estudio 06 · Fase 6 · Líneas base de otra familia (lectura, no decide)

Preregistro `PHASE6` (commit `71f446a`, aprobado por Rosa tal cual). Base: L + 5a. Misma matriz C3 (sin clima),
mismos orígenes desde 2005, mismos bloques y cortes, probabilidad NB2 con α dentro de la muestra → Platt por
horizonte con 1e-6 → p ≥ 0,50, y el conteo con la mezcla 5a.

- **6a:** GLM NB2 por horizonte (statsmodels), offset log(hist_lag_a + 0,5), columnas estandarizadas.
- **6b:** LightGBM Poisson por horizonte, init_score = margen de L, hiperparámetros de la V1 traducidos.

Ninguna familia se adopta: los veredictos son lectura. La 6′ (con clima) queda para cuando llegue la 1b.


In [1]:
import json
import sys

import lightgbm
import numpy as np
import pandas as pd
import statsmodels

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.model import tracking

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase1_history as p1
import phase3_onsets as p3
import phase5_counts as p5
import phase6_other_families as p6
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase6.json"
versions = {"lightgbm": lightgbm.__version__, "statsmodels": statsmodels.__version__}
inp = p3.phase3_inputs()
versions


{'lightgbm': '4.7.0', 'statsmodels': '0.15.0'}

## 1 · Base L + 5a y guarda G16 (reproducción de fase5.json)


In [2]:
base = p6.l_plus_5a(inp["serie"])
saved5 = json.loads((STUDY / "docs" / "metricas" / "fase5.json").read_text(encoding="utf-8"))
g16_repro = p6.guard_g16_reproduction(base["evaluaciones"], saved5["veredictos"]["5a"]["metricas"])
if not g16_repro["pasa"]:
    raise RuntimeError(f"Falla G16 (reproducción de L + 5a): se detiene la fase. {g16_repro}")
g16_repro


{'diferencia_maxima': 4.9451664985156185e-11, 'filas': 18, 'pasa': True}

## 2 · Familias (5 semillas; el GLM es determinista) y guardas G14 y G16


In [3]:
runs = {f: p6.run_family(inp, f, pr.SEEDS) for f in p6.FAMILIES}
l_alphas = p5.read_l_alphas()
g14 = {f: p6.guard_g14(r["ajustes"], l_alphas) for f, r in runs.items()}
again = p6.predict_blocks_family(inp["matrices"], inp["origenes"], p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_LONG),
                                 17, p3._config()[0], "6b_lightgbm")
again.attrs.pop("ajustes")
first = runs["6b_lightgbm"]["predicciones"][17]
g16_det = {"diferencia_maxima": float(np.abs(again.casos_predichos.to_numpy() - first.casos_predichos.to_numpy()).max()),
           "mismas_filas": len(again) == len(first)}
g16_det["pasa"] = g16_det["mismas_filas"] and g16_det["diferencia_maxima"] == 0.0
if not (all(g["pasa"] for g in g14.values()) and g16_det["pasa"]):
    raise RuntimeError(f"Falla una guarda: se detiene la fase. G14={g14} G16={g16_det}")
{"G14": g14, "G16_determinismo": g16_det}


{'G14': {'6a_nb_glm': {'mismos_bloques': True,
   'bloques_distintos': 0,
   'pasa': True},
  '6b_lightgbm': {'mismos_bloques': True,
   'bloques_distintos': 0,
   'pasa': True}},
 'G16_determinismo': {'diferencia_maxima': 0.0,
  'mismas_filas': True,
  'pasa': True}}

In [4]:
conv = runs["6a_nb_glm"]["ajustes"]
conv_cols = [c for c in conv.columns if c.startswith("convergio")]
print("Bloques del GLM sin convergencia:", int((~conv[conv_cols].all(axis=1)).sum()), "de", len(conv))
conv[["bloque", "filas_entrenamiento", *conv_cols]]


Bloques del GLM sin convergencia: 18 de 18


,bloque,filas_entrenamiento,convergio_h2,convergio_h3,convergio_h4
0,temporada_2008,8645,False,False,True
1,temporada_2009,12025,True,False,True
2,temporada_2010,15470,False,False,False
3,temporada_2011,18850,True,False,False
4,temporada_2012,22230,False,False,False
5,temporada_2013,25610,False,False,False
6,temporada_2014,28990,False,True,False
7,temporada_2015,32370,True,True,False
8,temporada_2016,35815,False,False,True
9,temporada_2017,39195,True,False,False


In [5]:
p6.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for f, r in runs.items():
    pd.concat(r["predicciones"].values(), ignore_index=True).to_csv(p6.OUTPUT_DIR / f"predicciones_{f}.csv.gz", index=False)
    r["ajustes"].drop(columns=["corte"]).to_csv(p6.OUTPUT_DIR / f"ajustes_{f}.csv", index=False)


## 3 · Lectura: veredictos con las reglas de las fases 3 (F1) y 5 (razón MAE), frente a L + 5a


In [6]:
f1_rule = {f: p3.compare(r, base) for f, r in runs.items()}
mae_rule = {f: p5.compare(r, base) for f, r in runs.items()}
mae_rule_mu = {f: p5.compare({"evaluaciones": r["evaluaciones_mu"]}, base) for f, r in runs.items()}
table = pd.concat({(f, regla): d["veredicto"][["horizonte", "veredicto", "vetos"]]
                   for regla, src in (("F1 (fase 3)", f1_rule), ("razón MAE (fase 5)", mae_rule),
                                      ("razón MAE, μ sin mezclar", mae_rule_mu))
                   for f, d in src.items()}, names=["familia", "regla"]).reset_index(level=[0, 1])
table


,familia,regla,horizonte,veredicto,vetos
0,6a_nb_glm,F1 (fase 3),2,empate,"brier, razon_mae"
1,6a_nb_glm,F1 (fase 3),3,pierde,"auprc, brier, razon_mae"
2,6a_nb_glm,F1 (fase 3),4,gana,NaN
0,6b_lightgbm,F1 (fase 3),2,empate,None
1,6b_lightgbm,F1 (fase 3),3,empate,None
2,6b_lightgbm,F1 (fase 3),4,empate,None
0,6a_nb_glm,razón MAE (fase 5),2,empate,"razon_rmse, brier"
1,6a_nb_glm,razón MAE (fase 5),3,empate,"razon_rmse, f1_050, auprc, brier"
2,6a_nb_glm,razón MAE (fase 5),4,empate,NaN
0,6b_lightgbm,razón MAE (fase 5),2,empate,None


In [7]:
pd.concat({f: d["antelacion"] for f, d in f1_rule.items()}, names=["familia"]).round(3)


horizonte  episodios  diferencia  ic95_inferior  ic95_superior  diferencia_temporada_2010  diferencia_temporada_2015  diferencia_temporada_2016  diferencia_temporada_2022  diferencia_temporada_2023  \
familia                                                                                                                                                                                                                
6a_nb_glm   0          2        193       0.036          0.000          0.073                      0.033                      0.000                      0.147                     -0.026                      0.035   
            1          3        193      -0.016         -0.073          0.041                     -0.067                      0.000                      0.176                      0.000                     -0.123   
            2          4        193       0.098          0.052          0.145                      0.000                      0.088                      0.294                      0.053                      0.070   
6b_lightgbm 0          2        193       0.000         -0.031          0.031                      0.033                     -0.029                     -0.029                      0.053                     -0.018   
            1          3        193      -0.010         -0.026          0.000                      0.000                      0.000                      0.000                      0.000                     -0.035   
            2          4        193       0.005         -0.016          0.026                      0.000                      0.000                      0.000                      0.026                      0.000   

               episodios_temporada_2010  episodios_temporada_2015  episodios_temporada_2016  episodios_temporada_2022  episodios_temporada_2023  
familia                                                                                                                                          
6a_nb_glm   0                        30                        34                        34                        38                        57  
            1                        30                        34                        34                        38                        57  
            2                        30                        34                        34                        38                        57  
6b_lightgbm 0                        30                        34                        34                        38                        57  
            1                        30                        34                        34                        38                        57  
            2                        30                        34                        34                        38                        57

In [8]:
cols = ["horizonte", "bloque", "positivos_cand", "f1_050_cand", "f1_050_ref", "auprc_cand", "auprc_ref",
        "brier_cand", "brier_ref"]
pd.concat({f: d["metricas"][cols] for f, d in f1_rule.items()}, names=["familia"]).round(3)


horizonte          bloque  positivos_cand  f1_050_cand  f1_050_ref  auprc_cand  auprc_ref  brier_cand  brier_ref
familia                                                                                                                         
6a_nb_glm   3           2  temporada_2010           234.0        0.710       0.722       0.788      0.780       0.031      0.030
            8           2  temporada_2015           557.0        0.774       0.772       0.886      0.885       0.052      0.051
            9           2  temporada_2016           395.0        0.661       0.657       0.750      0.757       0.062      0.059
            15          2  temporada_2022           431.0        0.708       0.712       0.779      0.781       0.054      0.054
            16          2  temporada_2023          1039.0        0.864       0.857       0.951      0.947       0.060      0.062
            17          2  temporada_2024           857.0        0.794       0.799       0.890      0.893       0.076      0.074
            21          3  temporada_2010           234.0        0.009       0.631       0.721      0.706       0.041      0.036
            26          3  temporada_2015           557.0        0.612       0.649       0.844      0.839       0.065      0.064
            27          3  temporada_2016           395.0        0.625       0.627       0.695      0.709       0.070      0.065
            33          3  temporada_2022           431.0        0.681       0.704       0.760      0.771       0.058      0.056
            34          3  temporada_2023          1039.0        0.742       0.835       0.859      0.930       0.106      0.073
            35          3  temporada_2024           857.0        0.773       0.788       0.865      0.876       0.085      0.079
            39          4  temporada_2010           234.0        0.522       0.476       0.640      0.600       0.041      0.042
            44          4  temporada_2015           557.0        0.648       0.480       0.821      0.811       0.067      0.075
            45          4  temporada_2016           395.0        0.545       0.590       0.598      0.660       0.092      0.070
            51          4  temporada_2022           431.0        0.656       0.652       0.725      0.743       0.062      0.060
            52          4  temporada_2023          1039.0        0.820       0.811       0.924      0.922       0.081      0.081
            53          4  temporada_2024           857.0        0.745       0.761       0.821      0.840       0.096      0.089
6b_lightgbm 3           2  temporada_2010           234.0        0.724       0.722       0.780      0.780       0.030      0.030
            8           2  temporada_2015           557.0        0.767       0.772       0.885      0.885       0.051      0.051
            9           2  temporada_2016           395.0        0.659       0.657       0.755      0.757       0.059      0.059
            15          2  temporada_2022           431.0        0.716       0.712       0.782      0.781       0.054      0.054
            16          2  temporada_2023          1039.0        0.857       0.857       0.948      0.947       0.062      0.062
            17          2  temporada_2024           857.0        0.800       0.799       0.894      0.893       0.073      0.074
            21          3  temporada_2010           234.0        0.631       0.631       0.702      0.706       0.036      0.036
            26          3  temporada_2015           557.0        0.641       0.649       0.840      0.839       0.065      0.064
            27          3  temporada_2016           395.0        0.628       0.627       0.707      0.709       0.065      0.065
            33          3  temporada_2022           431.0        0.710       0.704       0.770      0.771       0.056      0.056
            34          3  temporada_2023          1039.0        0.835       0.835       0.930      0.930       0.073      0.073
            35   

In [9]:
cols = ["horizonte", "bloque", "positivos_cand", "razon_mae_cand", "razon_mae_ref", "razon_rmse_cand", "razon_rmse_ref"]
pd.concat({f: d["metricas"][cols] for f, d in mae_rule.items()}, names=["familia"]).round(3)


horizonte          bloque  positivos_cand  razon_mae_cand  razon_mae_ref  razon_rmse_cand  razon_rmse_ref
familia                                                                                                                  
6a_nb_glm   3           2  temporada_2010           234.0           1.000          0.907            1.000           0.899
            8           2  temporada_2015           557.0           1.000          0.889            1.000           0.932
            9           2  temporada_2016           395.0           1.000          1.030            0.995           0.995
            15          2  temporada_2022           431.0           0.952          0.946            0.960           0.965
            16          2  temporada_2023          1039.0           0.849          0.808            0.919           0.893
            17          2  temporada_2024           857.0           1.039          1.136            1.081           1.316
            21          3  temporada_2010           234.0           1.000          0.891            1.000           0.890
            26          3  temporada_2015           557.0           1.000          0.828            1.000           0.914
            27          3  temporada_2016           395.0           1.000          1.042            1.000           1.016
            33          3  temporada_2022           431.0           1.000          0.927            1.000           0.990
            34          3  temporada_2023          1039.0           1.000          0.763            1.000           0.852
            35          3  temporada_2024           857.0           1.000          1.238            1.000           1.485
            39          4  temporada_2010           234.0           0.814          0.876            0.843           0.859
            44          4  temporada_2015           557.0           0.744          0.793            0.834           0.880
            45          4  temporada_2016           395.0           1.427          1.062            1.390           1.033
            51          4  temporada_2022           431.0           0.943          0.929            1.009           1.038
            52          4  temporada_2023          1039.0           0.711          0.733            0.790           0.815
            53          4  temporada_2024           857.0           1.368          1.362            1.474           1.690
6b_lightgbm 3           2  temporada_2010           234.0           0.896          0.907            0.892           0.899
            8           2  temporada_2015           557.0           0.896          0.889            0.929           0.932
            9           2  temporada_2016           395.0           1.015          1.030            0.976           0.995
            15          2  temporada_2022           431.0           0.941          0.946            0.954           0.965
            16          2  temporada_2023          1039.0           0.798          0.808            0.867           0.893
            17          2  temporada_2024           857.0           1.116          1.136            1.302           1.316
            21          3  temporada_2010           234.0           0.915          0.891            0.897           0.890
            26          3  temporada_2015           557.0           0.826          0.828            0.898           0.914
            27          3  temporada_2016           395.0           1.031          1.042            0.991           1.016
            33          3  temporada_2022           431.0           0.932          0.927            0.980           0.990
            34          3  temporada_2023          1039.0           0.764          0.763            0.841           0.852
            35          3  temporada_2024           857.0           1.256          1.238            1.537           1.485
            39          4  temporada_2010           234.0           0.871          0.876            0.862       

## 4 · Lecturas complementarias: sin 2010 y w de la mezcla por familia


In [10]:
pd.concat({(f, regla): d["lectura_sin_2010"][["horizonte", "veredicto"]]
           for regla, src in (("F1", f1_rule), ("razón MAE", mae_rule)) for f, d in src.items()},
          names=["familia", "regla"]).reset_index(level=[0, 1])


,familia,regla,horizonte,veredicto
0,6a_nb_glm,F1,2,empate
1,6a_nb_glm,F1,3,pierde
2,6a_nb_glm,F1,4,gana
0,6b_lightgbm,F1,2,empate
1,6b_lightgbm,F1,3,empate
2,6b_lightgbm,F1,4,empate
0,6a_nb_glm,razón MAE,2,empate
1,6a_nb_glm,razón MAE,3,empate
2,6a_nb_glm,razón MAE,4,empate
0,6b_lightgbm,razón MAE,2,empate


In [11]:
w = pd.concat({f: pd.concat({s: e["w"] for s, e in r["evaluaciones"].items()}, names=["semilla"]).reset_index(level=0)
               for f, r in runs.items()}, names=["familia"]).reset_index(level=0)
w.groupby(["familia", "horizonte"]).w.agg(["median", "min", "max"]).round(2)


median   min  max
familia     horizonte                   
6a_nb_glm   2            0.41  0.00  1.0
            3            0.00  0.00  1.0
            4            0.76  0.60  1.0
6b_lightgbm 2            0.71  0.53  1.0
            3            0.74  0.56  1.0
            4            0.81  0.60  1.0

## 5 · Salidas: métricas y MLflow


In [12]:
to_json = lambda d: {k: diag.to_json(v) for k, v in d.items() if isinstance(v, pd.DataFrame)}
content = {
    "versiones": versions, "guardas": {"G14": g14, "G16_determinismo": g16_det, "G16_reproduccion": g16_repro},
    "convergencia_glm": diag.to_json(conv.drop(columns=["corte"])),
    "lectura_regla_f1": {f: to_json(d) for f, d in f1_rule.items()},
    "lectura_regla_mae": {f: to_json(d) for f, d in mae_rule.items()},
    "lectura_regla_mae_mu_sin_mezclar": {f: to_json(d) for f, d in mae_rule_mu.items()},
    "w_5a": diag.to_json(w.drop(columns=["corte"])),
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdict_text = lambda d: "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in d["veredicto"].itertuples())
tracking.log_study(
    "mejora_modelo_fase6_otras_familias", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "6", "preregistro": "71f446a", **versions,
                       **{f"f1.{f}": verdict_text(d) for f, d in f1_rule.items()},
                       **{f"mae.{f}": verdict_text(d) for f, d in mae_rule.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={f"veredicto_{f}": table[table.familia.eq(f)] for f in p6.FAMILIES},
    description="Fase 6 del estudio 06: GLM NB2 y LightGBM con la matriz y el protocolo de L (lectura).")


C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio 637310f706b84c19b6856cbd37552e6a en 'denguekay/mejora_modelo_fase6_otras_familias' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'637310f706b84c19b6856cbd37552e6a'